In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


# Inisialisasi spark


In [ ]:
!pip install pyspark -q

# 2. Import Library Utama
import os
import tensorflow as tf
from pyspark.sql import SparkSession

# 3. Verifikasi Akselerasi GPU
print("Daftar Perangkat Fisik GPU:")
print(tf.config.list_physical_devices('GPU'))
print("-" * 40)

# 4. Inisialisasi Apache Spark dengan Konfigurasi Teruji (Anti-OOM)
print("Menginisialisasi Spark Session (Alokasi Terkendali Sesuai Skenario 2W)...")

spark = SparkSession.builder \
    .appName("SkinCancer_BigData_Irisan3Kelas") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.executor.memoryOverhead", "384m") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .getOrCreate()

print("Spark Session Berhasil Dibuat (Alokasi Aman ~4 GB RAM)!")


# Load dataset


In [ ]:
import os
import time
import glob
import warnings
import pandas as pd
from pathlib import Path
from pyspark.sql.functions import col, when
from pyspark import TaskContext

warnings.filterwarnings('ignore')
spark.sparkContext.setLogLevel("ERROR")

print("[INFO] Memproses Dataset Riset Irisan 3 Kelas (HAM10000 + ISIC 2019 + ISIC 2017)...")

# 1. Pencarian File Manifest Master Irisan 3 Kelas (dari pipeline sk1)
csv_candidates = [
    Path("/kaggle/input/dataset-irisan-multiclass-final/dataset_irisan_multiclass_final.csv"),
    Path("/kaggle/input/dataset_irisan_multiclass_final.csv"),
    Path("Dataset/csv_irisan_3kelas/dataset_irisan_multiclass_final.csv"),
    Path("Dataset/dataset_irisan_multiclass_final.csv"),
    Path("../../Dataset/dataset_irisan_multiclass_final.csv"),
    Path("/kaggle/working/dataset_irisan_multiclass_final.csv")
]

csv_path = None
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = str(candidate)
        break

if not csv_path:
    for root in ["/kaggle/input", "Dataset", "."]:
        if os.path.exists(root):
            matches = glob.glob(f"{root}/**/dataset_irisan_multiclass_final.csv", recursive=True)
            if matches:
                csv_path = matches[0]
                break

# 2. Pemuatan Dataset Menggunakan Apache Spark
if csv_path and os.path.exists(csv_path):
    print(f"[INFO] Memuat manifest irisan dari: {csv_path}")
    spark_df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
    
    # Harmonisasi label ke 3 kelas standar riset
    target_col = "harmonized_label" if "harmonized_label" in spark_df_raw.columns else "label"
    spark_df = spark_df_raw.withColumn(
        "label",
        when(col(target_col) == "NV", "nevus")
        .when(col(target_col) == "MEL", "melanoma")
        .when(col(target_col) == "BKL", "seborrheic_keratosis")
        .when(col(target_col) == "nevus", "nevus")
        .when(col(target_col) == "melanoma", "melanoma")
        .when(col(target_col) == "seborrheic_keratosis", "seborrheic_keratosis")
        .otherwise("other")
    ).filter(col("label") != "other")
    
    # Pelacakan Waktu Executor Spark untuk Laporan Big Data
    def track_executor_time(iterator):
        start_time = time.time()
        count = sum(1 for _ in iterator)
        elapsed = time.time() - start_time
        ctx = TaskContext.get()
        pid = ctx.partitionId() if ctx else "Unknown"
        yield (pid, count, elapsed)

    metrik_executors = spark_df.rdd.mapPartitions(track_executor_time).collect()
    print("\n=== LAPORAN BEBAN KERJA EXECUTOR SPARK (IRISAN 3 KELAS) ===")
    for pid, count, elapsed in metrik_executors:
        print(f"Executor Task [Partisi {pid}] -> Memproses {count} baris dalam {elapsed:.4f} detik")
    print("============================================================\n")
    
    selected_cols = [c for c in ["image_id", "label", "source", "filepath", "split", "lesion_id"] if c in spark_df.columns]
    df_final = spark_df.select(*selected_cols).toPandas()
else:
    print("[WARNING] Manifest CSV tidak ditemukan, memindai direktori gambar Kaggle...")
    ham_meta = '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
    if not os.path.exists(ham_meta):
        ham_meta = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'
    
    df_ham = pd.read_csv(ham_meta)
    df_ham = df_ham[df_ham['dx'].isin(['nv', 'mel', 'bkl'])].copy()
    label_map = {'nv': 'nevus', 'mel': 'melanoma', 'bkl': 'seborrheic_keratosis'}
    df_ham['label'] = df_ham['dx'].map(label_map)
    df_ham['source'] = 'ham10000'
    df_final = df_ham[['image_id', 'label', 'source']].copy()

# 3. Penyesuaian Path Citra Fisik di Lingkungan Eksekusi (Kaggle / Lokal)
print("[INFO] Mengindeks path citra fisik pada direktori input...")
image_index = {}
search_roots = ["/kaggle/input", "Dataset", "."]

for s_root in search_roots:
    if os.path.exists(s_root):
        for root, _, files in os.walk(s_root):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    stem = os.path.splitext(f)[0]
                    if stem not in image_index:
                        image_index[stem] = os.path.join(root, f)

def resolve_path(row):
    raw_p = str(row.get('filepath', ''))
    if raw_p and os.path.exists(raw_p):
        return raw_p
    img_id = str(row['image_id'])
    if img_id in image_index:
        return image_index[img_id]
    return raw_p

df_final['filepath'] = df_final.apply(resolve_path, axis=1)

# Validasi Keberadaan Berkas Fisik
tersedia = df_final['filepath'].apply(os.path.exists).sum()
print(f"[INFO] Indeks citra siap: {tersedia} dari {len(df_final)} berkas fisik terverifikasi.")

# 4. Tutup Spark Session untuk Melepas Memori RAM ke Sistem
spark.stop()
import gc
gc.collect()
print("[INFO] Spark Session ditutup, seluruh memori RAM dibebaskan untuk tahap pelatihan TensorFlow.")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Menampilkan Ringkasan Distribusi Data Irisan 3 Kelas
print("=== RINGKASAN DATASET IRISAN (HAM10000 ∩ ISIC 2017 ∩ ISIC 2019) ===")
print(f"Total Citra Terdaftar: {len(df_final)} citra\n")

print("1. Distribusi Berdasarkan Kelas Medis:")
kelas_counts = df_final['label'].value_counts()
for k, v in kelas_counts.items():
    persen = (v / len(df_final)) * 100
    print(f"   - {k:<22}: {v:6d} citra ({persen:.2f}%)")

if 'source' in df_final.columns:
    print("\n2. Distribusi Berdasarkan Sumber Dataset:")
    src_counts = df_final['source'].value_counts()
    for s, v in src_counts.items():
        print(f"   - {s:<22}: {v:6d} citra")

# Visualisasi Distribusi Kelas
plt.figure(figsize=(8, 4))
bars = plt.bar(kelas_counts.index, kelas_counts.values, color=['#2b5c8f', '#d95f02', '#7570b3'])
plt.title("Distribusi Citra Dataset Irisan 3 Kelas (Bebas Duplikat)", fontsize=12)
plt.ylabel("Jumlah Citra")
plt.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 150, f"{yval:,}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


# preprosesing citra


In [ ]:
# =============================================================================
# TAHAP 3: REKAYASA CITRA (224x224 RESNET-50)
# PIPELINE: PAD & RESIZE 224 -> DULL RAZOR -> EXACT CONTOUR SEGMENTATION -> SOFT-BLEND
# (Masker menjiplak 100% bentuk asli penyakit, warna alami murni tanpa Contrast Stretching)
# =============================================================================
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

TARGET_SIZE = 224
MIN_LESION_RATIO = 0.03
MAX_LESION_RATIO = 0.85

def pad_and_resize(img_rgb, target_size=TARGET_SIZE):
    h, w = img_rgb.shape[:2]
    max_dim = max(h, w)
    top, bottom = (max_dim - h) // 2, (max_dim - h) - ((max_dim - h) // 2)
    left, right = (max_dim - w) // 2, (max_dim - w) - ((max_dim - w) // 2)
    padded = cv2.copyMakeBorder(img_rgb, top, bottom, left, right, cv2.BORDER_REFLECT)
    return cv2.resize(padded, (target_size, target_size), interpolation=cv2.INTER_AREA)

def remove_hair_dull_razor(img_rgb, threshold=10):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(img_rgb, hair_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)

def segment_lesion_exact_contour(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    closed = cv2.morphologyEx(thresh_mask, cv2.MORPH_CLOSE, kernel)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return np.ones(gray.shape, dtype=np.uint8) * 255, False
    c = max(cnts, key=cv2.contourArea)
    total_area = gray.shape[0] * gray.shape[1]
    area = cv2.contourArea(c)
    mask = np.zeros(gray.shape, dtype=np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    is_success = (MIN_LESION_RATIO * total_area <= area <= MAX_LESION_RATIO * total_area)
    return mask, is_success

def soft_blend(img_rgb, mask):
    alpha = cv2.GaussianBlur(mask.astype(np.float32) / 255.0, (15, 15), 0)[:, :, np.newaxis]
    blurred_bg = cv2.GaussianBlur(img_rgb, (21, 21), 0)
    return (alpha * img_rgb + (1.0 - alpha) * blurred_bg).astype(np.uint8)

def preprocess_exact(img_rgb):
    resized = pad_and_resize(img_rgb, target_size=TARGET_SIZE)
    hairless = remove_hair_dull_razor(resized, threshold=10)
    mask, is_success = segment_lesion_exact_contour(hairless)
    final_img = soft_blend(hairless, mask) if is_success else hairless
    return resized, hairless, mask, final_img

def plot_perbandingan(asli, resized, hairless, mask, segmented, index):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(asli); axes[0].set_title(f"1. Asli (Sampel {index})"); axes[0].axis('off')
    axes[1].imshow(resized); axes[1].set_title("2. Resize (224x224)"); axes[1].axis('off')
    axes[2].imshow(hairless); axes[2].set_title("3. Dull Razor (No Hair)"); axes[2].axis('off')
    axes[3].imshow(mask, cmap='gray'); axes[3].set_title("4. Mask Kontur Asli"); axes[3].axis('off')
    axes[4].imshow(segmented); axes[4].set_title("5. Soft-Blended ROI"); axes[4].axis('off')
    plt.tight_layout()
    plt.show()

output_dir = "/kaggle/working/processed_images/"
os.makedirs(output_dir, exist_ok=True)

print("[INFO] Menampilkan visualisasi 3 sampel pertama (Exact Contour 224x224):")
for index, row in df_final.head(3).iterrows():
    img_bgr = cv2.imread(row['filepath'])
    if img_bgr is not None:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        resized, hairless, mask, final_img = preprocess_exact(img_rgb)
        plot_perbandingan(img_rgb, resized, hairless, mask, final_img, index)

def process_single_image(args):
    index, img_path = args
    filename = f"{index}_{os.path.basename(img_path)}"
    new_img_path = os.path.join(output_dir, filename)
    if os.path.exists(new_img_path):
        return index, new_img_path
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return index, img_path
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    _, _, _, final_img = preprocess_exact(img_rgb)
    cv2.imwrite(new_img_path, cv2.cvtColor(final_img, cv2.COLOR_RGB2BGR))
    return index, new_img_path

tasks = [(idx, row['filepath']) for idx, row in df_final.iterrows()]
num_workers = min(os.cpu_count() or 4, 8)
print(f"[INFO] Memproses {len(tasks)} citra secara paralel menggunakan {num_workers} worker CPU...")

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(tqdm(executor.map(process_single_image, tasks), total=len(tasks)))

results.sort(key=lambda x: x[0])
df_final['filepath'] = [r[1] for r in results]
df_final.to_csv("/kaggle/working/processed_metadata_combined.csv", index=False)
print(f"\n[INFO] Selesai! Sebanyak {len(results)} citra tersegmentasi kontur riil siap digunakan.")


## cek hasil


In [ ]:
import cv2
import matplotlib.pyplot as plt
samples = df_final.head(5)

print("=== Pengecekan Dimensi 5 Gambar Teratas ===")
plt.figure(figsize=(15, 4))
for i, (index, row) in enumerate(samples.iterrows()):
    img_path = row['filepath']
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tinggi, lebar, _ = img.shape
    nama_file = os.path.basename(img_path)
    print(f"[{nama_file}] -> Lebar: {lebar}, Tinggi: {tinggi}")
    plt.subplot(1, 5, i + 1)
    plt.title(f"Sampel {i+1}\n{lebar}x{tinggi}")
    plt.imshow(img)
    plt.axis('off')

plt.tight_layout()
plt.show()


# split dataset


In [ ]:
# =============================================================================
# TAHAP 4: PEMISAHAN DATA BEBAS KEBOCORAN & BALANCED DATA GENERATOR
# =============================================================================
import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Menggunakan Partisi Anti-Kebocoran Lesi (Lesion-Aware Stratified Split 80:10:10)
if 'split' in df_final.columns and set(df_final['split'].unique()).issuperset({'train', 'val', 'test'}):
    print("[INFO] Menggunakan pembagian partisi Lesion-Aware resmi (100% Bebas Kebocoran Lesi Pasien):")
    train_df = df_final[df_final['split'] == 'train'].copy()
    val_df = df_final[df_final['split'] == 'val'].copy()
    test_df = df_final[df_final['split'] == 'test'].copy()
else:
    print("[INFO] Melakukan pembagian stratified 70:15:15...")
    train_df, temp_df = train_test_split(df_final, test_size=0.30, random_state=42, stratify=df_final['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print("=== RINGKASAN PEMBAGIAN DATA ===")
print(f"Jumlah data Train      : {len(train_df)} ({len(train_df)/len(df_final)*100:.2f}%)")
print(f"Jumlah data Validation : {len(val_df)} ({len(val_df)/len(df_final)*100:.2f}%)")
print(f"Jumlah data Test       : {len(test_df)} ({len(test_df)/len(df_final)*100:.2f}%)\n")

# 2. OVERSAMPLING SEMUA KELAS MINORITAS PADA DATA LATIH (TRAIN SET SAJA)
max_count = train_df['label'].value_counts().max()
dfs_balanced = []

for kelas in sorted(train_df['label'].unique()):
    df_kelas = train_df[train_df['label'] == kelas]
    if len(df_kelas) < max_count:
        df_kelas_upsampled = resample(
            df_kelas,
            replace=True,
            n_samples=max_count,
            random_state=42
        )
        dfs_balanced.append(df_kelas_upsampled)
    else:
        dfs_balanced.append(df_kelas)

train_df_balanced = pd.concat(dfs_balanced).sample(frac=1, random_state=42).reset_index(drop=True)

print("=== DISTRIBUSI TRAIN SET (BALANCED OVERSAMPLED) ===")
print(train_df_balanced['label'].value_counts().to_string())
print("\n====================================================\n")

# 3. KONFIGURASI AUGMENTASI TINGKAT LANJUT (JURNAL LA-CapsNet)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,             # Rotasi -15 s/d +15 derajat
    brightness_range=[0.7, 1.3],   # Pengaturan kecerahan & kontras
    horizontal_flip=True,          # Flip horizontal acak
    vertical_flip=True,            # Flip vertikal acak
    zoom_range=0.15,               # Random zoom / crop
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='reflect'
)

# Validation dan Test HANYA di-preprocess, TIDAK di-augmentasi
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# 4. PEMBUATAN BATCH DATA
target_dim = (224, 224)
batch_sz = 32

train_data = train_datagen.flow_from_dataframe(
    dataframe=train_df_balanced,
    x_col="filepath",
    y_col="label",
    target_size=target_dim,
    batch_size=batch_sz,
    class_mode="categorical",
    shuffle=True
)

val_data = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="label",
    target_size=target_dim,
    batch_size=batch_sz,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    target_size=target_dim,
    batch_size=batch_sz,
    class_mode="categorical",
    shuffle=False
)


## cek hasil


In [ ]:
# =============================================================================
# CEK HASIL GENERATOR TAHAP 4 & SPESIFIKASI BATCH
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np

# 1. Mengambil 1 batch data dari generator
images, labels = next(iter(train_data))

# 2. Mengambil pemetaan nama kelas
class_labels = list(train_data.class_indices.keys())

# 3. Menampilkan Spesifikasi Teknis
print("=== SPESIFIKASI BATCH GAMBAR TAHAP 4 ===")
print(f"Bentuk Array Batch   : {images.shape} -> (Batch Size, Tinggi, Lebar, Channel RGB)")
print(f"Resolusi per Gambar  : {images.shape[1]} x {images.shape[2]} piksel")
print(f"Bentuk Label Batch   : {labels.shape} -> (Batch Size, Jumlah Kelas)")
print(f"Tipe Data Array      : {images.dtype}")
print(f"Rentang Nilai Piksel : {np.min(images):.2f} sampai {np.max(images):.2f} (Preprocess ResNet-50 BGR Aktif)")
print(f"Nama Kelas           : {class_labels}")
print("========================================\n")

# 4. Visualisasi 5 Gambar Pertama (Balikkan Preprocess ResNet50 agar RGB Alami)
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i in range(5):
    ax = axes[i]
    img_show = images[i].copy()
    img_show = img_show + [103.939, 116.779, 123.68]  # Re-add ImageNet mean
    img_show = img_show[..., ::-1]                    # BGR -> RGB
    img_show = np.clip(img_show, 0, 255) / 255.0

    label_idx = int(np.argmax(labels[i]))
    nama_kelas = class_labels[label_idx]
    ax.imshow(img_show)
    ax.set_title(f"Gambar {i+1}\nKelas: {nama_kelas}\n(Label: {label_idx})", fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()


# Modeling


In [ ]:
# =============================================================================
# TAHAP 5: KONSTRUKSI ARSITEKTUR MODEL (MULTI-GPU) + TRIPLET ATTENTION
# =============================================================================
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
    BatchNormalization, Dropout, Concatenate, Input,
    Conv2D, Activation, Lambda, Multiply
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K

# ---- TRIPLET ATTENTION MODULE (WACV 2021) ----
class ZPool(tf.keras.layers.Layer):
    def call(self, x):
        max_out = tf.reduce_max(x, axis=-1, keepdims=True)
        avg_out = tf.reduce_mean(x, axis=-1, keepdims=True)
        return tf.concat([max_out, avg_out], axis=-1)

class AttentionGate(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.zpool = ZPool()
        self.conv = Conv2D(1, kernel_size=3, padding='same', use_bias=False)
        self.sigmoid = Activation('sigmoid')

    def call(self, x):
        out = self.zpool(x)
        out = self.conv(out)
        out = self.sigmoid(out)
        return out

class TripletAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate_ch = AttentionGate()
        self.gate_cw = AttentionGate()
        self.gate_hw = AttentionGate()

    def call(self, x):
        # Branch 1: C x H
        x_perm1 = tf.transpose(x, perm=[0, 3, 2, 1])
        attn1 = self.gate_ch(x_perm1)
        x_perm1 = tf.transpose(x_perm1 * attn1, perm=[0, 3, 2, 1])

        # Branch 2: C x W
        x_perm2 = tf.transpose(x, perm=[0, 1, 3, 2])
        attn2 = self.gate_cw(x_perm2)
        x_perm2 = tf.transpose(x_perm2 * attn2, perm=[0, 1, 3, 2])

        # Branch 3: H x W
        attn3 = self.gate_hw(x)
        x_branch3 = x * attn3

        return (x_perm1 + x_perm2 + x_branch3) / 3.0

# ---- BANGUN MODEL DI DALAM SCOPE MULTI-GPU MIRROREDSTRATEGY ----
strategy = tf.distribute.MirroredStrategy()
print(f"[INFO] Jumlah GPU Tersinkronisasi: {strategy.num_replicas_in_sync}")

with strategy.scope():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    # Triplet Attention disematkan pada conv4_block6_out (resolusi 14x14x1024)
    conv4_out = base_model.get_layer('conv4_block6_out').output
    attended = TripletAttention(name='triplet_attention')(conv4_out)
    attended_avg = GlobalAveragePooling2D(name='att_avg_pool')(attended)

    # Fitur utama ResNet50 output (7x7x2048)
    x_main = base_model.output
    avg_pool = GlobalAveragePooling2D()(x_main)
    max_pool = GlobalMaxPooling2D()(x_main)

    # Fusi fitur multi-skala
    x = Concatenate()([avg_pool, max_pool, attended_avg])
    x = BatchNormalization()(x)

    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    # Output 3 Kelas Medis (Softmax)
    predictions = Dense(3, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

print("Ringkasan Arsitektur Model (Multi-GPU + Triplet Attention @ Conv4 14x14 - 3 Kelas):")
model.summary()


# Training


In [ ]:
# =============================================================================
# TAHAP 6: PELATIHAN 2 FASE (MULTI-GPU, FOCAL LOSS, MICRO-UNFREEZE)
# =============================================================================
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

# 1. METRIK, FOCAL LOSS & CALLBACKS
focal_loss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0,
    alpha=0.25
)

metrics_list = [
    'accuracy',
    tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=3),
    tf.keras.metrics.AUC(curve='PR', name='pr_auc', multi_label=True, num_labels=3)
]

early_stop = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    mode='min',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath='/kaggle/working/best_melanoma_model.keras',
    monitor='val_pr_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

callbacks_list = [lr_scheduler, early_stop, checkpoint]

# 2. FASE 1: WARM-UP (Melatih Head Saja, Base Model Frozen)
base_model.trainable = False

with strategy.scope():
    model.compile(
        optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
        loss=focal_loss,
        metrics=metrics_list
    )

print("[INFO] Memulai Fase 1: Warm-up Custom Head...")
start_time_f1 = time.time()

history_f1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_list
)

waktu_f1 = (time.time() - start_time_f1) / 60
print(f"Waktu Pelatihan Fase 1: {waktu_f1:.2f} menit\n")

# 3. FASE 2: MICRO-UNFREEZING (HANYA conv5_block3, BatchNorm Dikunci)
print("[INFO] Membuka conv5_block3, Kunci Seluruh Layer BatchNorm...")

with strategy.scope():
    base_model.trainable = True
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        elif "conv5_block3" in layer.name:
            layer.trainable = True
        else:
            layer.trainable = False

    model.compile(
        optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4),
        loss=focal_loss,
        metrics=metrics_list
    )

print("[INFO] Memulai Fase 2: Micro Fine-Tuning...")
start_time_f2 = time.time()

history_f2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=callbacks_list
)

waktu_f2 = (time.time() - start_time_f2) / 60
print(f"Waktu Pelatihan Fase 2: {waktu_f2:.2f} menit\n")

# 4. EVALUASI BLIND TEST PADA DATA UJI (TEST SET)
print(f"[HASIL] Total Waktu Pelatihan: {waktu_f1+waktu_f2:.2f} menit")
print("\n[INFO] Menjalankan Evaluasi Blind Test...")
eval_results = model.evaluate(test_data)

final_loss = eval_results[0]
final_accuracy = eval_results[1]
final_auc = eval_results[2]
final_pr_auc = eval_results[3]

print(f"\n--- HASIL BLIND TEST AKHIR ---")
print(f"Akurasi Test : {final_accuracy * 100:.2f}%")
print(f"Loss Test    : {final_loss:.4f}")
print(f"AUC          : {final_auc:.4f}")
print(f"PR-AUC       : {final_pr_auc:.4f}")
print("------------------------------\n")


# Evaluasi


In [ ]:
# =============================================================================
# TAHAP 7: EVALUASI MODEL & VISUALISASI KOMPREHENSIF (3-KELAS)
# =============================================================================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. MENGGABUNGKAN RIWAYAT PELATIHAN DARI FASE 1 & FASE 2
acc = history_f1.history['accuracy'] + history_f2.history['accuracy']
val_acc = history_f1.history['val_accuracy'] + history_f2.history['val_accuracy']
loss = history_f1.history['loss'] + history_f2.history['loss']
val_loss = history_f1.history['val_loss'] + history_f2.history['val_loss']

batas_fase = len(history_f1.history['accuracy']) - 1

# 2. VISUALISASI GRAFIK AKURASI DAN LOSS
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=batas_fase, color='red', linestyle='--', label='Mulai Fine-Tuning')
plt.title('Pergerakan Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=batas_fase, color='red', linestyle='--', label='Mulai Fine-Tuning')
plt.title('Pergerakan Loss Model')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 3. EVALUASI VALIDATION SET (3-KELAS)
print("\n[INFO] Memuat model terbaik dari checkpoint fisik...")
best_model = tf.keras.models.load_model(
    '/kaggle/working/best_melanoma_model.keras',
    custom_objects={
        'TripletAttention': TripletAttention,
        'AttentionGate': AttentionGate,
        'ZPool': ZPool,
    }
)

print("[INFO] Mengevaluasi Validation Set...")
val_data.reset()
Y_pred_val = best_model.predict(val_data, steps=len(val_data), verbose=0)
y_pred_classes_val = np.argmax(Y_pred_val, axis=1)
y_true_val = np.array(val_data.classes)
target_names = list(val_data.class_indices.keys())
y_pred_classes_val = y_pred_classes_val[:len(y_true_val)]

print("=== CLASSIFICATION REPORT (VALIDATION SET) ===")
print(classification_report(y_true_val, y_pred_classes_val, target_names=target_names))

# 4. BLIND TEST & EVALUASI TEST SET
print("\n[INFO] Mengevaluasi Test Set (Blind Test)...")
test_data.reset()
Y_pred_test = best_model.predict(test_data, steps=len(test_data), verbose=0)
y_pred_classes_test = np.argmax(Y_pred_test, axis=1)
y_true_test = np.array(test_data.classes)
y_pred_classes_test = y_pred_classes_test[:len(y_true_test)]

print("=== CLASSIFICATION REPORT (TEST SET) ===")
print(classification_report(y_true_test, y_pred_classes_test, target_names=target_names))

# 5. VISUALISASI CONFUSION MATRIX BERDAMPINGAN (3x3)
cm_val = confusion_matrix(y_true_val, y_pred_classes_val)
cm_test = confusion_matrix(y_true_test, y_pred_classes_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12}, ax=axes[0])
axes[0].set_title('Confusion Matrix - Validation Set', fontsize=14)
axes[0].set_ylabel('Label Aktual', fontsize=12)
axes[0].set_xlabel('Label Prediksi', fontsize=12)

sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12}, ax=axes[1])
axes[1].set_title('Confusion Matrix - Test Set', fontsize=14)
axes[1].set_ylabel('Label Aktual', fontsize=12)
axes[1].set_xlabel('Label Prediksi', fontsize=12)

plt.tight_layout()
plt.show()
